# Tool Evaluation with Promptfoo

Promptfoo now owns the evaluation loop. The notebook prepares tests from `evaluation.xml`, runs a Python provider that wraps the tool-calling agent, and reads Promptfoo's JSONL results.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import subprocess
import sys
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
EVALUATION_DIR = PROJECT_ROOT / "notebooks" / "evals"
EVALUATION_FILE = EVALUATION_DIR / "evaluation.xml"
PROVIDER_FILE = EVALUATION_DIR / "promptfoo_agent_provider.py"
PROMPTFOO_CONFIG_FILE = EVALUATION_DIR / "promptfooconfig.json"
PROMPTFOO_RESULTS_FILE = EVALUATION_DIR / "promptfoo-results.jsonl"

load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation directory: {EVALUATION_DIR}")
print(f"Evaluation file: {EVALUATION_FILE}")
print(f"Promptfoo provider: {PROVIDER_FILE}")

## Promptfoo flow

```text
evaluation.xml
    |
    v
notebook builds promptfooconfig.json
    |
    v
npx promptfoo eval
    |
    v
promptfoo_agent_provider.py
    |
    v
Azure Responses API + calculator tool
    |
    v
promptfoo-results.jsonl + markdown report
```

> 💡 Promptfoo is an evaluation runner for LLM apps. It handles test cases, providers, assertions, result exports, and the web viewer, so this notebook does not need a hand-rolled eval loop.

## Load evaluation tasks

In [ ]:
def parse_evaluation_file(file_path: Path) -> list[dict[str, str]]:
    """Parse the trusted local XML evaluation file."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    evaluations: list[dict[str, str]] = []

    for task in root.findall(".//task"):
        prompt_elem = task.find("prompt")
        response_elem = task.find("response")
        if prompt_elem is None or response_elem is None:
            continue
        evaluations.append(
            {
                "prompt": (prompt_elem.text or "").strip(),
                "response": (response_elem.text or "").strip(),
            }
        )

    return evaluations


tasks = parse_evaluation_file(EVALUATION_FILE)
print(f"Loaded {len(tasks)} tasks")
tasks

## Build Promptfoo config

The provider is a Promptfoo Python provider. Promptfoo calls `call_api(prompt, options, context)` in `promptfoo_agent_provider.py` for each test case.

In [ ]:
def build_promptfoo_config(tasks: list[dict[str, str]]) -> dict[str, Any]:
    return {
        "$schema": "https://promptfoo.dev/config-schema.json",
        "description": "Calculator tool evaluation via Promptfoo",
        "prompts": ["{{prompt}}"],
        "providers": [
            {
                "id": f"file://{PROVIDER_FILE.name}",
                "label": "azure-tool-agent",
                "config": {"max_turns": 12},
            }
        ],
        "tests": [
            {
                "description": f"Task {index + 1}",
                "vars": {
                    "prompt": task["prompt"],
                    "expected": task["response"],
                },
                "assert": [
                    {
                        "type": "equals",
                        "value": task["response"],
                    }
                ],
            }
            for index, task in enumerate(tasks)
        ],
        "evaluateOptions": {
            "maxConcurrency": 1,
            "showProgressBar": True,
            "cache": False,
        },
    }


promptfoo_config = build_promptfoo_config(tasks)
PROMPTFOO_CONFIG_FILE.write_text(json.dumps(promptfoo_config, indent=2), encoding="utf-8")
print(f"Wrote {PROMPTFOO_CONFIG_FILE}")

In [ ]:
promptfoo_config["tests"][:2]

## Run Promptfoo

This uses `npx --yes promptfoo@latest eval` so the notebook does not need a JavaScript package checked into the repo.

In [ ]:
def run_promptfoo(
    config_file: Path = PROMPTFOO_CONFIG_FILE,
    results_file: Path = PROMPTFOO_RESULTS_FILE,
) -> Path:
    if not PROVIDER_FILE.exists():
        raise FileNotFoundError(f"Missing provider file: {PROVIDER_FILE}")

    if results_file.exists():
        results_file.unlink()

    command = [
        "npx",
        "--yes",
        "promptfoo@latest",
        "eval",
        "--config",
        str(config_file),
        "--output",
        str(results_file),
        "--no-cache",
    ]

    env_file = PROJECT_ROOT / ".env"
    if env_file.exists():
        command.extend(["--env-file", str(env_file)])

    env = os.environ.copy()
    env["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{env.get('PATH', '')}"
    env.setdefault("PROMPTFOO_DISABLE_TELEMETRY", "1")

    print("Running:", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=EVALUATION_DIR,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )

    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()

    return results_file

In [ ]:
results_path = run_promptfoo()
results_path

## Read results and build a compact report

In [ ]:
def load_promptfoo_jsonl(results_file: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with results_file.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def _row_response(row: dict[str, Any]) -> dict[str, Any]:
    response = row.get("response") or {}
    return response if isinstance(response, dict) else {"output": response}


def _row_vars(row: dict[str, Any]) -> dict[str, Any]:
    if isinstance(row.get("vars"), dict):
        return row["vars"]
    test_case = row.get("testCase") or row.get("test") or {}
    if isinstance(test_case, dict) and isinstance(test_case.get("vars"), dict):
        return test_case["vars"]
    return {}


def build_markdown_report(rows: list[dict[str, Any]], tasks: list[dict[str, str]]) -> str:
    total = len(rows)
    correct = sum(1 for row in rows if row.get("success") is True)
    accuracy = (correct / total * 100) if total else 0

    durations = []
    tool_call_counts = []
    sections = [
        "# Evaluation Report",
        "",
        "## Summary",
        "",
        f"- **Accuracy**: {correct}/{total} ({accuracy:.1f}%)",
    ]

    for index, row in enumerate(rows):
        response = _row_response(row)
        metadata = response.get("metadata") or {}
        duration = metadata.get("durationSeconds")
        if isinstance(duration, int | float):
            durations.append(duration)
        tool_call_count = metadata.get("numToolCalls", 0)
        if isinstance(tool_call_count, int | float):
            tool_call_counts.append(tool_call_count)

    if durations:
        sections.append(f"- **Average Task Duration**: {sum(durations) / len(durations):.2f}s")
    if tool_call_counts:
        sections.append(f"- **Average Tool Calls per Task**: {sum(tool_call_counts) / len(tool_call_counts):.2f}")
        sections.append(f"- **Total Tool Calls**: {int(sum(tool_call_counts))}")

    sections.append("\n---")

    for index, row in enumerate(rows):
        response = _row_response(row)
        metadata = response.get("metadata") or {}
        vars_ = _row_vars(row)
        fallback_task = tasks[index] if index < len(tasks) else {"prompt": "", "response": ""}
        prompt = vars_.get("prompt") or fallback_task["prompt"]
        expected = vars_.get("expected") or fallback_task["response"]
        actual = response.get("output")
        passed = row.get("success") is True
        duration = metadata.get("durationSeconds")
        tool_calls = metadata.get("toolCalls", {})
        summary = metadata.get("summary") or "N/A"
        feedback = metadata.get("feedback") or "N/A"

        sections.extend(
            [
                "",
                "### Task",
                "",
                f"**Prompt**: {prompt}",
                f"**Ground Truth Response**: `{expected}`",
                f"**Actual Response**: `{actual}`",
                f"**Correct**: {'✅' if passed else '❌'}",
                f"**Duration**: {duration:.2f}s" if isinstance(duration, int | float) else "**Duration**: N/A",
                "**Tool Calls**:",
                "```json",
                json.dumps(tool_calls, indent=2),
                "```",
                "",
                "**Summary**",
                summary,
                "",
                "**Feedback**",
                feedback,
                "",
                "---",
            ]
        )

    return "\n".join(sections)

In [ ]:
rows = load_promptfoo_jsonl(PROMPTFOO_RESULTS_FILE)
report = build_markdown_report(rows, tasks)
print(report)

## Optional: open Promptfoo UI

After running the eval, you can inspect the latest results in Promptfoo's web UI:

```bash
cd notebooks
npx promptfoo@latest view
```